# Data Overview

This notebook explores the raw Lending Club loan dataset to understand its structure, key fields, missing values, and potential signals related to loan repayment risk.


## Raw Dataset Inspection

The raw dataset contains one or more CSV files representing historical loan records. Initial inspection focuses on understanding file size, available columns, and potential label fields without loading the entire dataset into memory.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

%matplotlib inline



In [ ]:
pd.set_option('display.max_columns', None)

DATA_RAW_PATH = "../data/raw/archive/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"
df = pd.read_csv(DATA_RAW_PATH, nrows=100000)

print(df)

### Dataset Size and Complexity

The accepted loans dataset contains approximately 151 columns, reflecting borrower attributes, loan terms, repayment behavior, and post-origination outcomes. Not all columns are suitable for modeling, and careful selection is required to avoid data leakage.


### Target Definition
In a loan / BNPL system, the decision is made at the time of application.

At that moment, the bank does not know the real outcome; it only has an estimate.

The actual outcome (repayment or default) is known only after time has passed, once the loan has been given.

When training a model, we learn from historical loans where the outcome is already known.

Therefore, the target variable must represent the real final outcome of past loans, because that reflects true user behavior and allows the model to learn meaningful patterns.

### Data Leakage
Any information created after the loan decision is made is not available at application time.

If such information is shown to the model during prediction, it breaks the real-world timeline.

Information like the final status of the loan directly reveals the outcome.

If the model already knows the outcome, it is no longer predicting anything.

Therefore, any information that directly or indirectly reveals the final result must never be available at prediction time.

In [30]:
columns_list = list(df.columns)
for column in columns_list:
    print(column)

id
member_id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
url
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
fico_range_low
fico_range_high
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
last_fico_range_high
last_fico_range_low
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
to

### Summary
Target:
The target represents the final outcome of a loan. In this project, the target is loan_status, which indicates whether the user eventually repaid the loan or defaulted.

Leakage:
Leakage includes any columns that should not be available to the model at the time of prediction. This includes information related to the final loan status, payment-related information, and default-related information, since these are only known after the loan has been issued.

Simple rule to avoid leakage:
Any information that would not be known at the time a user applies for a loan should not be given to the model during prediction.

### Feature Availability Filtering 

### Columns to Exclude 
- loan_status
- total_pymnt_inv
- total_rec_prncp
- total_rec_int
- total_rec_late_fee
- last_pymnt_d
- last_pymnt_amnt
- next_pymnt_d
- settlement_status
- settlement_date
- settlement_amount
- settlement_percentage
- settlement_term

This is an initial, conservative list of leakage columns and will be refined further during feature engineering and model evaluation.





In [ ]:
df.shape

In [ ]:
df.head(50)

In [ ]:

df.columns

In [ ]:
cols = list(df.columns)
cols

In [ ]:
df['loan_status'].value_counts()

In [ ]:
#filter dataset to only keep finalized loan outcomes for modeling

loan_status_list = ['Fully Paid', 'Charged Off', 'Default']
df_filtered = df[df['loan_status'].isin(loan_status_list)].copy() #create copy because its independent of the original dataframe and pandas does not get confused

df_filtered['loan_status'].value_counts()

In [ ]:
#mapping values of fully paid (0), charged off (1), default(1) in new column df_filtered[is_default]. The new column is also the target variable for
#the model. 

status_to_map = {
    'Fully Paid':0,
    'Charged Off':1,
    'Default':1
}

df_filtered['is_default'] = df_filtered['loan_status'].map(status_to_map)

df_filtered[['loan_status', 'is_default']].head(50)

### ## Feature Analysis & Data Leakage Handling

### Objective

The goal of this step is to identify which features should be used for training the model and which should be excluded to prevent data leakage and improve model reliability.

---

### Key Principle

A feature should only be used if it is available at **prediction time**.

> Any feature that contains information from the future or reflects the outcome of the loan introduces **data leakage** and must be removed.

---

### Target Variable

We defined the target variable as:

* `is_default = 1` → borrower defaulted (`Charged Off`, `Default`)
* `is_default = 0` → borrower did not default (`Fully Paid`)

---

### Feature Evaluation

#### 1. Useful Features (Kept)

* `loan_amnt` → represents total loan burden
* `term` → affects repayment duration and exposure
* `int_rate` → reflects borrower risk profile
* `installment` → represents monthly repayment burden
* `annual_inc` → indicates repayment capacity

These features are available at prediction time and are directly or indirectly related to default risk.

---

#### 2. Engineered Feature Insight

We identified that:

* `installment / annual_inc` → captures **repayment stress**

This feature provides a stronger signal by combining income and repayment burden.

---

#### 3. Dropped Features (Data Leakage)

The following features were removed because they contain **post-loan behavior**:

* `total_pymnt` → total amount paid over time
* `last_pymnt_d` → last payment date
* `next_pymnt_d` → next scheduled payment date (dynamic, reflects repayment status)

These features are not available at prediction time and indirectly reveal the loan outcome.

---

### Key Takeaways

* Features must represent information available at the time of prediction
* Avoid using features derived from future events
* Both raw and engineered features can be useful
* Understanding feature meaning is critical for building reliable models

---

### Conclusion

After this step, we have:

* A clean dataset with valid features
* No data leakage
* A well-defined target variable

This prepares the dataset for further preprocessing and modeling.


In [ ]:
cols_to_drop = ['total_pymnt', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
                'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d']

df_filtered = df_filtered.drop(columns=cols_to_drop) #cannot drop columns that are already dropped



In [ ]:
df_filtered.columns

### Exploratory Data Analysis 
### 'loan_amnt'

In [ ]:
df_filtered['loan_amnt'].hist()

In [ ]:
q1 = df_filtered['loan_amnt'].quantile(0.25)
q3 = df_filtered['loan_amnt'].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - (1.5 * iqr)
upper_bound =  q3 + (1.5 * iqr)

print(upper_bound)

### The upper bound is 38150.0 which tells us that the values 30k-35k are not statistical outliers and represent real-world loan amounts.
### Removing them would discard meaningful information for high-value loans. 

### EDA 
### annual_inc

In [ ]:
df_filtered['annual_inc'].hist()

### EDA for annual_inc
### The annual_inc feature is highly right-skewed.

### Most values are concentrated in the lower range, while a few very large values stretch the distribution and make it hard to visualize.

### To address this, we apply a log transformation to compress large values and better understand the distribution without removing data.

In [ ]:
df_filtered['log_annual_inc'] = np.log1p(df_filtered['annual_inc'])

In [ ]:
df_filtered['log_annual_inc'].hist()

### EDA
### instalment

In [ ]:
df_filtered['installment'].hist()

### EDA
### term

In [ ]:
df_filtered['term'].value_counts()

### EDA
### int_rate

In [ ]:
df_filtered['int_rate'].hist()

In [ ]:
missing_values = df_filtered.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
df_filtered.dtypes

In [25]:
missing_percentage = (missing_values / df_filtered.shape[0]) * 100
print(missing_percentage)

id                         0.000000
member_id                100.000000
loan_amnt                  0.000000
funded_amnt                0.000000
funded_amnt_inv            0.000000
                            ...    
settlement_amount         96.677741
settlement_percentage     96.677741
settlement_term           96.677741
is_default                 0.000000
log_annual_inc             0.000000
Length: 145, dtype: float64


In [27]:
high_missing = missing_percentage[missing_percentage > 70]
high_missing.shape

(55,)

In [28]:
cols_to_drop = high_missing.index

df_filtered = df_filtered.drop(columns=cols_to_drop)

In [29]:
df_filtered.shape

(87892, 90)

In [37]:
df_filtered['term']  = df['term']
df_filtered['term'].unique()

array([' 36 months', ' 60 months'], dtype=object)

In [38]:
df_filtered['term'] = df_filtered['term'].str.strip()
df_filtered['term'].unique()



array(['36 months', '60 months'], dtype=object)

In [39]:
term_mapping = {
    '36 months' : 0,
    '60 months' : 1
}

df_filtered['term'] = df_filtered['term'].map(term_mapping)

df_filtered['term'].unique()

array([0, 1])

In [41]:
df_filtered['grade'].unique()

array(['C', 'B', 'F', 'A', 'E', 'D', 'G'], dtype=object)

In [44]:
df_filtered['grade'] = df['grade']

In [45]:
grade_mapping = {
    'A' : 0,
    'B' : 1,
    'C' : 2,
    'D' : 3,
    'E' : 4,
    'F' : 5,
    'G' : 6
}

df_filtered['grade'] = df_filtered['grade'].map(grade_mapping)

df_filtered['grade'].unique()

array([2, 1, 5, 0, 4, 3, 6])

In [46]:
df_filtered['sub_grade'].unique()

array(['C4', 'C1', 'B4', 'F1', 'C3', 'B2', 'B1', 'A2', 'B5', 'C2', 'E2',
       'A4', 'E3', 'C5', 'A1', 'D4', 'F3', 'D1', 'B3', 'D3', 'D5', 'A5',
       'F2', 'E4', 'D2', 'E1', 'F5', 'E5', 'A3', 'G2', 'G1', 'G3', 'G4',
       'F4', 'G5'], dtype=object)

In [47]:
sub_grades = df_filtered['sub_grade'].unique()

sorted_sub_grades = sorted(sub_grades)

sub_grade_mapping = {}

for index,value in enumerate(sorted_sub_grades):
    sub_grade_mapping[value] = index

print(sub_grade_mapping)

{'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'B1': 5, 'B2': 6, 'B3': 7, 'B4': 8, 'B5': 9, 'C1': 10, 'C2': 11, 'C3': 12, 'C4': 13, 'C5': 14, 'D1': 15, 'D2': 16, 'D3': 17, 'D4': 18, 'D5': 19, 'E1': 20, 'E2': 21, 'E3': 22, 'E4': 23, 'E5': 24, 'F1': 25, 'F2': 26, 'F3': 27, 'F4': 28, 'F5': 29, 'G1': 30, 'G2': 31, 'G3': 32, 'G4': 33, 'G5': 34}


In [49]:
df_filtered['sub_grade'] = df_filtered['sub_grade'].map(sub_grade_mapping)



In [50]:
df_filtered['sub_grade'].unique()

array([13, 10,  8, 25, 12,  6,  5,  1,  9, 11, 21,  3, 22, 14,  0, 18, 27,
       15,  7, 17, 19,  4, 26, 23, 16, 20, 29, 24,  2, 31, 30, 32, 33, 28,
       34])